# Subcategory-based demand forecasting

Selected original implementation, cleaned for public review. No commercial data or historical outputs are included. Run only with authorized local inputs. Configuration values are examples, not validated business settings.

In [ ]:
from pathlib import Path
import os

# Run from the repository root or notebooks directory.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
for folder in ['Input', 'Clustering', 'PreEstimation', 'Modeling', 'SubCatStrategy/Input', 'SubCatStrategy/Data', 'SubCatStrategy/SubCatForecast']:
    Path("workspace", folder).mkdir(parents=True, exist_ok=True)

## Sales and hierarchy preparation

In [ ]:
# Cell 1 — build Data.xlsx (function name must be Data)
import os
import calendar
from datetime import date, timedelta

import numpy as np
import pandas as pd

def Data(
    monthly_sales_path: str = r"workspace/SubCatStrategy/Input/monthly_sales.xlsx",
    subcat_path: str       = r"workspace/SubCatStrategy/Input/SubCat.xlsx",
    output_path: str       = r"workspace/SubCatStrategy/Data/Data.xlsx",
) -> pd.DataFrame:
    """
    Builds the SubCatStrategy modeling input file 'Data.xlsx' with the agreed schema:
    - Keys, hierarchy, sales measures
    - price_per_box
    - sub-category aggregates and share_vol
    - HijriMain_1..12 and HijriSec_1..12 dummy columns
    """

    # -----------------------------
    # 1) Read inputs (strict columns)
    # -----------------------------
    ms = pd.read_excel(monthly_sales_path)
    sc = pd.read_excel(subcat_path)

    ms_expected = ["year", "Month", "Material Code", "Sku Name", "Vol (Kg)", "Val($)", "Box", "RSP($)"]
    sc_expected = ["Code", "SKU Name (English)", "Sub Cat"]

    missing_ms = [c for c in ms_expected if c not in ms.columns]
    missing_sc = [c for c in sc_expected if c not in sc.columns]
    if missing_ms:
        raise ValueError(f"monthly_sales is missing columns: {missing_ms}. Found: {list(ms.columns)}")
    if missing_sc:
        raise ValueError(f"SubCat file is missing columns: {missing_sc}. Found: {list(sc.columns)}")

    # -----------------------------
    # 2) Clean + map sub-cat
    # -----------------------------
    ms = ms.copy()
    sc = sc.copy()

    ms["Material Code"] = ms["Material Code"].astype(str).str.strip()
    sc["Code"] = sc["Code"].astype(str).str.strip()

    # Map: Material Code -> Sub Cat (drop dupes defensively)
    sc_map = sc[["Code", "Sub Cat"]].drop_duplicates(subset=["Code"]).copy()
    ms = ms.merge(sc_map, how="left", left_on="Material Code", right_on="Code")

    # If unmapped, keep label as the SKU code to avoid silent data loss
    ms["Sub Cat"] = ms["Sub Cat"].where(ms["Sub Cat"].notna(), ms["Material Code"])

    # -----------------------------
    # 3) Type casting and derived metrics
    # -----------------------------
    ms["year"] = pd.to_numeric(ms["year"], errors="coerce").astype("Int64")
    # Month can be numeric or text; attempt numeric first
    month_num = pd.to_numeric(ms["Month"], errors="coerce")
    if month_num.notna().mean() > 0.8:
        ms["month"] = month_num.astype("Int64")
    else:
        # Parse text month names by using pandas datetime
        tmp = pd.to_datetime(ms["year"].astype(str) + "-" + ms["Month"].astype(str) + "-01", errors="coerce")
        if tmp.isna().mean() > 0.2:
            raise ValueError("Could not parse 'Month' into numeric month. Please ensure Month is 1..12 or standard month names.")
        ms["month"] = tmp.dt.month.astype("Int64")

    # Build canonical monthly date
    ms["date"] = pd.to_datetime(
        dict(year=ms["year"].astype(int), month=ms["month"].astype(int), day=1),
        errors="coerce"
    )
    ms = ms.dropna(subset=["date", "year", "month", "Material Code"]).copy()

    # Numeric columns
    for c in ["Vol (Kg)", "Val($)", "Box", "RSP($)"]:
        ms[c] = pd.to_numeric(ms[c], errors="coerce")

    ms["vol_kg"]  = ms["Vol (Kg)"].fillna(0.0).astype(float)
    ms["val_usd"] = ms["Val($)"].fillna(0.0).astype(float)
    ms["box"]     = ms["Box"].fillna(0.0).astype(float)
    ms["rsp_usd"] = ms["RSP($)"].astype(float)

    # price_per_box = val_usd / box, but only if box > 0
    ms["price_per_box"] = np.where(ms["box"] > 0, ms["val_usd"] / ms["box"], np.nan)

    # -----------------------------
    # 4) Sub-category aggregates + share target
    # -----------------------------
    ms = ms.rename(columns={
        "Sub Cat": "sub_cat",
        "Material Code": "material_code",
        "Sku Name": "sku_name",
    })

    grp_keys = ["date", "sub_cat"]
    subcat_agg = (
        ms.groupby(grp_keys, as_index=False)
          .agg(subcat_vol_kg=("vol_kg", "sum"),
               subcat_val_usd=("val_usd", "sum"),
               subcat_box=("box", "sum"))
    )
    ms = ms.merge(subcat_agg, on=grp_keys, how="left")

    # share_vol = vol_kg / subcat_vol_kg (safe)
    ms["share_vol"] = np.where(ms["subcat_vol_kg"] > 0, ms["vol_kg"] / ms["subcat_vol_kg"], 0.0)

    # -----------------------------
    # 5) Hijri dummies (HijriMain_1..12 and HijriSec_1..12)
    #     Main = hijri month with max overlap days in the Gregorian month
    #     Sec  = second most overlap (if exists)
    # -----------------------------
    # Converter adapters
    def _get_hijri_month_for_gregorian(y: int, m: int, d: int) -> int:
        """
        Returns Hijri month (1..12) for given Gregorian date.
        Tries hijridate first, then hijri_converter. Raises if neither exists.
        """
        try:
            from hijridate import Gregorian  # pip: hijridate
            return Gregorian(y, m, d).to_hijri().month
        except Exception:
            try:
                from hijri_converter import Gregorian as HG  # pip: hijri-converter
                return HG(y, m, d).to_hijri().month
            except Exception as e:
                raise ImportError(
                    "Hijri conversion library not found. Install one of these packages:\n"
                    "  - hijridate\n"
                    "  - hijri-converter\n"
                    "Then re-run this cell.\n"
                    f"Original error: {repr(e)}"
                )

    def _hijri_main_sec_for_month(y: int, m: int):
        last_day = calendar.monthrange(y, m)[1]
        counts = {k: 0 for k in range(1, 13)}
        for dd in range(1, last_day + 1):
            hm = _get_hijri_month_for_gregorian(y, m, dd)
            counts[int(hm)] += 1
        # sort by overlap days desc
        ranked = sorted(counts.items(), key=lambda kv: kv[1], reverse=True)
        main_month = ranked[0][0]
        # secondary only if a different month has >0 overlap
        sec_month = None
        for mm, ct in ranked[1:]:
            if ct > 0:
                sec_month = mm
                break
        return main_month, sec_month

    # Compute once per unique Gregorian month in the dataset
    uniq_months = (
        ms[["year", "month"]]
        .drop_duplicates()
        .sort_values(["year", "month"])
        .astype(int)
        .itertuples(index=False, name=None)
    )

    hijri_map = {}
    for y, m in uniq_months:
        hijri_map[(y, m)] = _hijri_main_sec_for_month(y, m)

    main_colnames = [f"HijriMain_{i}" for i in range(1, 13)]
    sec_colnames  = [f"HijriSec_{i}"  for i in range(1, 13)]
    for c in main_colnames + sec_colnames:
        ms[c] = 0

    # Assign dummies
    years_int = ms["year"].astype(int).values
    months_int = ms["month"].astype(int).values

    main_vals = np.zeros(len(ms), dtype=int)
    sec_vals  = np.zeros(len(ms), dtype=int)
    for i in range(len(ms)):
        main_m, sec_m = hijri_map[(int(years_int[i]), int(months_int[i]))]
        main_vals[i] = int(main_m)
        sec_vals[i] = int(sec_m) if sec_m is not None else 0

    for k in range(1, 13):
        ms[f"HijriMain_{k}"] = (main_vals == k).astype(int)
        ms[f"HijriSec_{k}"]  = (sec_vals == k).astype(int)

    # -----------------------------
    # 6) Final output schema + save
    # -----------------------------
    out = pd.DataFrame({
        "date": ms["date"].dt.strftime("%Y-%m-%d"),
        "year": ms["year"].astype(int),
        "month": ms["month"].astype(int),
        "sub_cat": ms["sub_cat"].astype(str),
        "material_code": ms["material_code"].astype(str),
        "sku_name": ms["sku_name"].astype(str),
        "vol_kg": ms["vol_kg"].astype(float),
        "val_usd": ms["val_usd"].astype(float),
        "box": ms["box"].astype(float),
        "rsp_usd": ms["rsp_usd"].astype(float),
        "price_per_box": ms["price_per_box"].astype(float),
        "subcat_vol_kg": ms["subcat_vol_kg"].astype(float),
        "subcat_val_usd": ms["subcat_val_usd"].astype(float),
        "subcat_box": ms["subcat_box"].astype(float),
        "share_vol": ms["share_vol"].astype(float),
    })

    # Append hijri dummies in required order
    for c in main_colnames:
        out[c] = ms[c].astype(int)
    for c in sec_colnames:
        out[c] = ms[c].astype(int)

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    out.to_excel(output_path, index=False)
    return out

## Model input and quality checks

In [ ]:
# Cell 1 — Build modeling-ready tables + QA checks (and ALWAYS refresh Data.xlsx by calling Data())

import os
import numpy as np
import pandas as pd

def Build_Model_Input(
    monthly_sales_path: str = r"workspace/SubCatStrategy/Input/monthly_sales.xlsx",
    subcat_path: str       = r"workspace/SubCatStrategy/Input/SubCat.xlsx",
    data_path: str         = r"workspace/SubCatStrategy/Data/Data.xlsx",
    model_input_path: str  = r"workspace/SubCatStrategy/Data/Model_Input.xlsx",
) -> dict:
    """
    Step 2 of the SubCatStrategy methodology (after defining Data schema):

    1) Refresh the base dataset by calling Data(...) and saving Data.xlsx.
    2) Create two modeling tables:
       - SubCat_TS:   one row per (date, sub_cat) -> target subcat_vol_kg + exog (Hijri dummies)
       - SKU_Share_TS: one row per (date, sub_cat, material_code) -> target share_vol + features

    3) Run QA checks to validate:
       - Sum(vol_kg by SKU) == subcat_vol_kg (per date, sub_cat)
       - Sum(share_vol by SKU) ≈ 1.0 (per date, sub_cat)
       - HijriMain dummies sum == 1; HijriSec dummies sum in {0,1}
       - Basic price_per_box sanity (outlier flagging)

    Outputs:
       - Excel file with sheets: SubCat_TS, SKU_Share_TS, QA_Checks
       - Returns a dict with DataFrames and summary metrics for convenience.
    """

    # ---------------------------------------------------------------------
    # A) Refresh base data (CRITICAL: always call Data() to update Data.xlsx)
    # ---------------------------------------------------------------------
    # NOTE: This assumes you already executed the cell where Data(...) is defined.
    df_data = Data(
        monthly_sales_path=monthly_sales_path,
        subcat_path=subcat_path,
        output_path=data_path,
    )

    # Ensure expected columns exist in Data output
    required = [
        "date", "year", "month", "sub_cat", "material_code", "sku_name",
        "vol_kg", "val_usd", "box", "rsp_usd", "price_per_box",
        "subcat_vol_kg", "subcat_val_usd", "subcat_box", "share_vol",
    ]
    missing = [c for c in required if c not in df_data.columns]
    if missing:
        raise ValueError(f"Data() output missing columns: {missing}. Found: {list(df_data.columns)}")

    # Identify Hijri dummy columns (must be present if Data() created them)
    hijri_main_cols = [f"HijriMain_{i}" for i in range(1, 13)]
    hijri_sec_cols  = [f"HijriSec_{i}"  for i in range(1, 13)]
    hijri_missing = [c for c in (hijri_main_cols + hijri_sec_cols) if c not in df_data.columns]
    if hijri_missing:
        raise ValueError(
            "Hijri dummy columns are missing from Data(). "
            "Please ensure Data() was built with HijriMain_1..12 and HijriSec_1..12.\n"
            f"Missing: {hijri_missing}"
        )

    # Parse date to datetime for correct sorting & grouping
    df = df_data.copy()
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date", "sub_cat", "material_code"]).copy()

    # ---------------------------------------------------------------------
    # B) Build modeling table 1: SubCat_TS (category-level time series)
    # ---------------------------------------------------------------------
    # One row per (date, sub_cat).
    # This is where you run ETS/Theta/SARIMA/SARIMAX to forecast category demand.
    subcat_ts = (
        df.groupby(["date", "sub_cat"], as_index=False)
          .agg(
              subcat_vol_kg=("subcat_vol_kg", "first"),   # constant within the group by construction
              subcat_val_usd=("subcat_val_usd", "first"),
              subcat_box=("subcat_box", "first"),
              year=("year", "first"),
              month=("month", "first"),
          )
          .sort_values(["sub_cat", "date"])
    )

    # Attach Hijri exogenous dummies at (date, sub_cat) level.
    # These are the same for all SKUs within a month, so taking "first" is safe.
    hijri_exog = (
        df.groupby(["date", "sub_cat"], as_index=False)[hijri_main_cols + hijri_sec_cols]
          .first()
    )
    subcat_ts = subcat_ts.merge(hijri_exog, on=["date", "sub_cat"], how="left")

    # Define the category model target explicitly (you can change later to val/box)
    subcat_ts = subcat_ts.rename(columns={"subcat_vol_kg": "y_subcat_vol_kg"})

    # ---------------------------------------------------------------------
    # C) Build modeling table 2: SKU_Share_TS (SKU share model input)
    # ---------------------------------------------------------------------
    # One row per (date, sub_cat, material_code).
    # This is where you model substitution using share_vol and drivers like price_per_box.
    sku_share_ts = df[[
        "date", "year", "month",
        "sub_cat", "material_code", "sku_name",
        "share_vol",
        "vol_kg", "val_usd", "box", "price_per_box",
        "subcat_vol_kg", "subcat_val_usd", "subcat_box",
    ] + hijri_main_cols + hijri_sec_cols].copy()

    # Optional: add a simple availability proxy for later modeling (no external inventory needed)
    # is_sold = 1 if volume>0 else 0
    sku_share_ts["is_sold"] = (sku_share_ts["vol_kg"] > 0).astype(int)

    # ---------------------------------------------------------------------
    # D) QA checks (these are your "gate" before any forecasting)
    # ---------------------------------------------------------------------

    # D1) Sum(vol_kg across SKUs) vs subcat_vol_kg (should match closely)
    vol_sum_check = (
        df.groupby(["date", "sub_cat"], as_index=False)
          .agg(
              sum_vol_kg=("vol_kg", "sum"),
              subcat_vol_kg=("subcat_vol_kg", "first"),
              n_skus=("material_code", "nunique")
          )
    )
    vol_sum_check["abs_diff_vol"] = (vol_sum_check["sum_vol_kg"] - vol_sum_check["subcat_vol_kg"]).abs()
    vol_sum_check["rel_diff_vol"] = np.where(
        vol_sum_check["subcat_vol_kg"].abs() > 0,
        vol_sum_check["abs_diff_vol"] / vol_sum_check["subcat_vol_kg"].abs(),
        0.0
    )

    # D2) Sum(share_vol across SKUs) should be ~1 (if subcat_vol_kg > 0)
    share_sum_check = (
        df.groupby(["date", "sub_cat"], as_index=False)
          .agg(
              share_sum=("share_vol", "sum"),
              subcat_vol_kg=("subcat_vol_kg", "first"),
              n_skus=("material_code", "nunique")
          )
    )
    share_sum_check["abs_diff_share"] = (share_sum_check["share_sum"] - 1.0).abs()
    # Only meaningful when category volume exists
    share_sum_check["is_relevant"] = (share_sum_check["subcat_vol_kg"] > 0).astype(int)

    # D3) Hijri dummy validity
    hijri_check = (
        df.groupby(["date"], as_index=False)[hijri_main_cols + hijri_sec_cols].first()
    )
    hijri_check["HijriMain_sum"] = hijri_check[hijri_main_cols].sum(axis=1)
    hijri_check["HijriSec_sum"]  = hijri_check[hijri_sec_cols].sum(axis=1)
    hijri_check["HijriMain_ok"]  = (hijri_check["HijriMain_sum"] == 1).astype(int)
    hijri_check["HijriSec_ok"]   = (hijri_check["HijriSec_sum"].isin([0, 1])).astype(int)

    # D4) price_per_box outlier flag (simple and interpretable)
    # Flag extreme values using robust percentiles (ignoring NaNs)
    ppb = sku_share_ts["price_per_box"].replace([np.inf, -np.inf], np.nan).dropna()
    if len(ppb) > 0:
        lo = float(ppb.quantile(0.005))
        hi = float(ppb.quantile(0.995))
    else:
        lo, hi = np.nan, np.nan

    sku_share_ts["price_per_box_outlier"] = 0
    if np.isfinite(lo) and np.isfinite(hi):
        sku_share_ts.loc[
            sku_share_ts["price_per_box"].notna() &
            ((sku_share_ts["price_per_box"] < lo) | (sku_share_ts["price_per_box"] > hi)),
            "price_per_box_outlier"
        ] = 1

    # Consolidated QA sheet (human-readable)
    qa_summary = pd.DataFrame([
        {"check": "Data rows", "value": len(df)},
        {"check": "Unique sub_cats", "value": df["sub_cat"].nunique()},
        {"check": "Unique SKUs", "value": df["material_code"].nunique()},
        {"check": "Date range start", "value": str(df["date"].min().date())},
        {"check": "Date range end", "value": str(df["date"].max().date())},
        {"check": "Vol sum check: max abs diff", "value": float(vol_sum_check["abs_diff_vol"].max())},
        {"check": "Vol sum check: max rel diff", "value": float(vol_sum_check["rel_diff_vol"].max())},
        {"check": "Share sum check: max abs diff (relevant only)", "value": float(share_sum_check.loc[share_sum_check["is_relevant"] == 1, "abs_diff_share"].max()) if (share_sum_check["is_relevant"]==1).any() else 0.0},
        {"check": "HijriMain validity (all dates ok?)", "value": int(hijri_check["HijriMain_ok"].min())},
        {"check": "HijriSec validity (all dates ok?)", "value": int(hijri_check["HijriSec_ok"].min())},
        {"check": "price_per_box outliers (rows flagged)", "value": int(sku_share_ts["price_per_box_outlier"].sum())},
        {"check": "price_per_box p0.5", "value": lo},
        {"check": "price_per_box p99.5", "value": hi},
    ])

    # If you want to inspect “problematic” rows quickly:
    # - large rel_diff_vol indicates mapping or aggregation issue
    vol_sum_issues = vol_sum_check.sort_values("rel_diff_vol", ascending=False).head(50)
    share_sum_issues = share_sum_check.sort_values("abs_diff_share", ascending=False).head(50)
    hijri_issues = hijri_check[(hijri_check["HijriMain_ok"] == 0) | (hijri_check["HijriSec_ok"] == 0)].copy()
    ppb_outliers = sku_share_ts[sku_share_ts["price_per_box_outlier"] == 1].copy()

    # ---------------------------------------------------------------------
    # E) Export to Excel (multi-sheet)
    # ---------------------------------------------------------------------
    os.makedirs(os.path.dirname(model_input_path), exist_ok=True)
    with pd.ExcelWriter(model_input_path, engine="openpyxl") as writer:
        subcat_ts.to_excel(writer, sheet_name="SubCat_TS", index=False)
        sku_share_ts.to_excel(writer, sheet_name="SKU_Share_TS", index=False)

        # QA checks and diagnostics
        qa_summary.to_excel(writer, sheet_name="QA_Checks", index=False, startrow=0)
        vol_sum_issues.to_excel(writer, sheet_name="QA_Checks", index=False, startrow=len(qa_summary) + 3)
        share_sum_issues.to_excel(writer, sheet_name="QA_Checks", index=False, startrow=len(qa_summary) + 3 + len(vol_sum_issues) + 3)
        hijri_issues.to_excel(writer, sheet_name="QA_Hijri_Issues", index=False)
        ppb_outliers.to_excel(writer, sheet_name="QA_PPB_Outliers", index=False)

    return {
        "data": df_data,
        "subcat_ts": subcat_ts,
        "sku_share_ts": sku_share_ts,
        "qa_summary": qa_summary,
        "model_input_path": model_input_path,
    }

## Subcategory forecasting and model selection

In [ ]:
# Cell 1 — UPDATED Step 3: Multi-metric selection -> choose Top-N -> FINAL forecast = median(Top-N)

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from math import sqrt

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing, SimpleExpSmoothing

# ---- optional sklearn ----
SKLEARN_OK = True
try:
    from sklearn.linear_model import Ridge
    from sklearn.preprocessing import StandardScaler
except Exception:
    SKLEARN_OK = False


# -------------------------
# Utilities
# -------------------------
def _coerce_month_start(s):
    dt = pd.to_datetime(s, errors="coerce")
    if not hasattr(dt, "dt"):
        dt = pd.to_datetime(pd.Series(dt), errors="coerce")
    return dt.dt.to_period("M").dt.start_time

def _make_future_dates(last_date, fsteps):
    last = pd.Timestamp(last_date)
    return pd.date_range(last + pd.offsets.MonthBegin(1), periods=int(fsteps), freq="MS")

def _future_exog_from_history(exog_hist, m, fsteps):
    """
    Repeat last seasonal block to create future exog (stable & leakage-safe).
    """
    X = np.asarray(exog_hist, dtype=float)
    if X.ndim == 1:
        X = X.reshape(-1, 1)
    if X.shape[0] == 0:
        return np.zeros((fsteps, X.shape[1]))
    block = X[-min(m, X.shape[0]):, :] if (m and m > 1) else X[-1:, :]
    reps = int(np.ceil(fsteps / block.shape[0]))
    Xf = np.vstack([block] * reps)[:fsteps, :]
    return np.where(np.isfinite(Xf), Xf, 0.0)

def _fallback_forecast(y, h):
    y = np.asarray(y, dtype=float)
    y = y[np.isfinite(y)]
    if len(y) == 0:
        return np.zeros(h, dtype=float)
    last = float(y[-1])
    mean = float(np.mean(y))
    val = last if np.isfinite(last) else mean
    if not np.isfinite(val):
        val = 0.0
    return np.maximum(np.repeat(val, h), 0.0)

def _metrics(y_true, y_pred, y_train_for_mase=None, m=1):
    """
    Multi-metric evaluation (complementary, not redundant):
      - MAPE: scale-free but unstable for small true values
      - sMAPE: more robust than MAPE in near-zero regimes
      - WAPE: volume-weighted absolute % error (good for planning)
      - RMSE: penalizes large misses
      - MAE: stable absolute error
      - MASE: relative to naive benchmark (useful for comparability)
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true, y_pred = y_true[mask], y_pred[mask]

    if len(y_true) == 0:
        return {"MAE": np.nan, "RMSE": np.nan, "MAPE": np.nan, "sMAPE": np.nan, "WAPE": np.nan, "MASE": np.nan}

    eps = 1e-9
    err = y_true - y_pred

    mae = float(np.mean(np.abs(err)))
    rmse = float(sqrt(np.mean(err**2)))

    mape = float(np.mean(np.abs(err) / np.maximum(np.abs(y_true), eps)) * 100.0)
    smape = float(np.mean(2.0 * np.abs(err) / np.maximum(np.abs(y_true) + np.abs(y_pred), eps)) * 100.0)
    wape = float(np.sum(np.abs(err)) / np.maximum(np.sum(np.abs(y_true)), eps) * 100.0)

    # MASE scale from training (seasonal naive if m>1 else naive)
    mase = np.nan
    if y_train_for_mase is not None:
        ytr = np.asarray(y_train_for_mase, dtype=float)
        ytr = ytr[np.isfinite(ytr)]
        if len(ytr) > (m if (m and m > 1) else 1):
            if m and m > 1:
                scale = np.mean(np.abs(ytr[m:] - ytr[:-m]))
            else:
                scale = np.mean(np.abs(ytr[1:] - ytr[:-1]))
            if np.isfinite(scale) and scale > 0:
                mase = float(mae / scale)

    return {"MAE": mae, "RMSE": rmse, "MAPE": mape, "sMAPE": smape, "WAPE": wape, "MASE": mase}

def _rank_weighted_score(df_metrics, weights):
    """
    Convert multiple metrics into a single, stable selection score.

    Why rank-based?
      - Works across different scales (RMSE vs MAPE)
      - Robust to a few extreme values
      - Easy to explain: "best average rank across metrics"
    """
    df = df_metrics.copy()

    # Ensure numeric
    for c in ["MAPE", "sMAPE", "WAPE", "RMSE", "MAE", "MASE"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # MASE may be NaN for short history; treat as worst rank by filling very large
    df["MASE_fill"] = df["MASE"].copy()
    if df["MASE_fill"].notna().any():
        worst = float(df["MASE_fill"].max())
        df["MASE_fill"] = df["MASE_fill"].fillna(worst + 1.0)
    else:
        df["MASE_fill"] = 1e9

    # Rank each metric (lower is better)
    df["r_MAPE"]  = df["MAPE"].rank(method="min", ascending=True)
    df["r_sMAPE"] = df["sMAPE"].rank(method="min", ascending=True)
    df["r_WAPE"]  = df["WAPE"].rank(method="min", ascending=True)
    df["r_RMSE"]  = df["RMSE"].rank(method="min", ascending=True)
    df["r_MAE"]   = df["MAE"].rank(method="min", ascending=True)
    df["r_MASE"]  = df["MASE_fill"].rank(method="min", ascending=True)

    # Weighted sum of ranks (lower is better)
    df["score"] = (
        weights["MAPE"]  * df["r_MAPE"]  +
        weights["sMAPE"] * df["r_sMAPE"] +
        weights["WAPE"]  * df["r_WAPE"]  +
        weights["RMSE"]  * df["r_RMSE"]  +
        weights["MAE"]   * df["r_MAE"]   +
        weights["MASE"]  * df["r_MASE"]
    )

    return df


# -------------------------
# Forecast models (same set as before)
# -------------------------
def _theta_forecast(y, h):
    y = np.asarray(y, dtype=float)
    n = len(y)
    if n < 3:
        return _fallback_forecast(y, h)

    t = np.arange(n, dtype=float)
    A = np.vstack([t, np.ones(n)]).T
    b1, b0 = np.linalg.lstsq(A, y, rcond=None)[0]

    t_f = np.arange(n, n + h, dtype=float)
    lin_fc = b1 * t_f + b0

    try:
        ses = SimpleExpSmoothing(y, initialization_method="estimated").fit(optimized=True)
        ses_fc = np.asarray(ses.forecast(h), dtype=float)
    except Exception:
        ses_fc = _fallback_forecast(y, h)

    return np.maximum(0.5 * (lin_fc + ses_fc), 0.0)

def _ets_forecast(y, h, m):
    y = np.asarray(y, dtype=float)
    if len(y) < 6:
        return _fallback_forecast(y, h)

    seasonal = "add" if (m and m > 1 and len(y) >= 2*m) else None
    try:
        fit = ExponentialSmoothing(
            y,
            trend="add",
            seasonal=seasonal,
            seasonal_periods=m if seasonal else None,
            initialization_method="estimated"
        ).fit(optimized=True)
        return np.maximum(np.asarray(fit.forecast(h), dtype=float), 0.0)
    except Exception:
        try:
            ses = SimpleExpSmoothing(y, initialization_method="estimated").fit(optimized=True)
            return np.maximum(np.asarray(ses.forecast(h), dtype=float), 0.0)
        except Exception:
            return _fallback_forecast(y, h)

def _sarimax_grid_search(y, X, m, d, D, seasonal=True):
    # small safe grid
    grid_p = (0, 1)
    grid_q = (0, 1)
    grid_P = (0, 1)
    grid_Q = (0, 1)
    trend_options = ("n", "c")

    best = None
    best_aic = np.inf

    y = np.asarray(y, dtype=float)
    X_use = None if X is None else np.asarray(X, dtype=float)

    for trend in trend_options:
        for p in grid_p:
            for q in grid_q:
                if seasonal:
                    for P in grid_P:
                        for Q in grid_Q:
                            order = (p, int(d), q)
                            seas  = (P, int(D), Q, int(m))
                            try:
                                res = SARIMAX(
                                    y, exog=X_use,
                                    order=order, seasonal_order=seas, trend=trend,
                                    enforce_stationarity=False, enforce_invertibility=False
                                ).fit(disp=False)
                                aic = float(res.aic)
                                if np.isfinite(aic) and aic < best_aic:
                                    best_aic = aic
                                    best = (order, seas, trend, aic)
                            except Exception:
                                pass
                else:
                    order = (p, int(d), q)
                    seas  = (0, 0, 0, 0)
                    try:
                        res = SARIMAX(
                            y, exog=X_use,
                            order=order, seasonal_order=seas, trend=trend,
                            enforce_stationarity=False, enforce_invertibility=False
                        ).fit(disp=False)
                        aic = float(res.aic)
                        if np.isfinite(aic) and aic < best_aic:
                            best_aic = aic
                            best = (order, seas, trend, aic)
                    except Exception:
                        pass

    return best

def _fit_forecast_sarimax(y, X, h, m, d, D, use_exog, seasonal):
    best = _sarimax_grid_search(y, X if use_exog else None, m, d, D, seasonal=seasonal)
    if best is None:
        return None
    order, seas, trend, _ = best
    try:
        mod = SARIMAX(
            np.asarray(y, dtype=float),
            exog=None if not use_exog else np.asarray(X, dtype=float),
            order=order, seasonal_order=seas, trend=trend,
            enforce_stationarity=False, enforce_invertibility=False
        )
        res = mod.fit(disp=False)
        if use_exog:
            Xf = _future_exog_from_history(X, m, h)
            fc = np.asarray(res.forecast(steps=h, exog=Xf), dtype=float)
        else:
            fc = np.asarray(res.forecast(steps=h), dtype=float)
        return np.maximum(fc, 0.0)
    except Exception:
        return None

def _regression_forecast(y, X, h, m):
    if not SKLEARN_OK:
        return None

    y = np.asarray(y, dtype=float)
    Xex = None if X is None else np.asarray(X, dtype=float)
    if Xex is not None and Xex.ndim == 1:
        Xex = Xex.reshape(-1, 1)

    n = len(y)
    if n < 12:
        return _fallback_forecast(y, h)

    max_lag = min(12, n - 2) if (m and m > 1) else min(6, n - 2)

    rows, tgt = [], []
    for t in range(max_lag, n):
        feats = [y[t - i] for i in range(1, max_lag + 1)]
        feats.append(np.mean(y[max(0, t - m):t]) if (m and m > 1) else np.mean(y[:t]))
        if Xex is not None:
            feats.extend(Xex[t].tolist())
        rows.append(feats)
        tgt.append(y[t])

    Xsup = np.asarray(rows, dtype=float)
    ysup = np.asarray(tgt, dtype=float)
    Xsup = np.where(np.isfinite(Xsup), Xsup, 0.0)
    ysup = np.where(np.isfinite(ysup), ysup, 0.0)

    scaler = StandardScaler()
    Xs = scaler.fit_transform(Xsup)

    model = Ridge(alpha=1.0, random_state=0)
    model.fit(Xs, ysup)

    y_hist = y.copy()
    Xf = None if Xex is None else _future_exog_from_history(Xex, m, h)

    fc = []
    for step in range(h):
        t = len(y_hist)
        feats = [y_hist[t - i] for i in range(1, max_lag + 1)]
        feats.append(np.mean(y_hist[max(0, t - m):t]) if (m and m > 1) else np.mean(y_hist))
        if Xf is not None:
            feats.extend(Xf[step].tolist())
        Xrow = np.asarray(feats, dtype=float).reshape(1, -1)
        Xrow = np.where(np.isfinite(Xrow), Xrow, 0.0)
        yhat = float(model.predict(scaler.transform(Xrow))[0])
        yhat = max(0.0, yhat)
        fc.append(yhat)
        y_hist = np.append(y_hist, yhat)

    return np.asarray(fc, dtype=float)

def _snaive_forecast(y, h, m):
    y = np.asarray(y, dtype=float)
    y = y[np.isfinite(y)]
    if len(y) == 0:
        return np.zeros(h, dtype=float)
    if not m or m <= 1 or len(y) < m:
        return _fallback_forecast(y, h)
    rep = y[-m:]
    reps = int(np.ceil(h / len(rep)))
    fc = np.tile(rep, reps)[:h]
    return np.maximum(fc, 0.0)


# -------------------------
# Main function
# -------------------------
def SubCatForecast(
    monthly_sales_path: str = r"workspace/SubCatStrategy/Input/monthly_sales.xlsx",
    subcat_path: str       = r"workspace/SubCatStrategy/Input/SubCat.xlsx",
    data_path: str         = r"workspace/SubCatStrategy/Data/Data.xlsx",
    model_input_path: str  = r"workspace/SubCatStrategy/Data/Model_Input.xlsx",

    seasonality: int = 12,
    fsteps_error: int = 6,
    fsteps_out: int = 6,

    # Ensemble size
    top_n_models: int = 3,

    # Multi-metric weights (rank-based score)
    # Interpretation: higher weight => metric influences selection more
    w_mape: float  = 3.0,
    w_smape: float = 2.0,
    w_wape: float  = 2.0,
    w_rmse: float  = 2.0,
    w_mae: float   = 1.0,
    w_mase: float  = 1.0,

    output_path: str = r"workspace/SubCatStrategy/SubCatForecast/SubCat_Forecasts.xlsx",
    verbose: bool = True,
) -> dict:
    """
    Sub-category forecasting (anchor forecast) with robust model selection.

    Selection logic (key station):
      1) Holdout last fsteps_error months.
      2) Compute metrics per model: MAPE, sMAPE, WAPE, RMSE, MAE, MASE.
      3) Build a weighted rank score over these metrics (lower is better).
      4) Choose Top-N models with lowest score.
      5) Refit each selected model on full history and forecast fsteps_out.
      6) FINAL forecast = median across selected-model forecasts.

    Important:
      - Always calls Build_Model_Input(...) first -> updates Model_Input.xlsx and Data.xlsx.
      - Keeps all model forecasts for transparency + audits.
    """

    # 1) Refresh upstream artifacts (Build_Model_Input -> Data)
    _ = Build_Model_Input(
        monthly_sales_path=monthly_sales_path,
        subcat_path=subcat_path,
        data_path=data_path,
        model_input_path=model_input_path,
    )

    # 2) Load SubCat_TS
    ts = pd.read_excel(model_input_path, sheet_name="SubCat_TS")
    if not {"date", "sub_cat", "y_subcat_vol_kg"}.issubset(ts.columns):
        raise ValueError("SubCat_TS must contain: date, sub_cat, y_subcat_vol_kg")

    ts["date"] = _coerce_month_start(ts["date"])
    ts = ts.dropna(subset=["date", "sub_cat"]).copy()
    ts = ts.sort_values(["sub_cat", "date"]).reset_index(drop=True)

    # Exog (Hijri dummies) if present
    hijri_main_cols = [f"HijriMain_{i}" for i in range(1, 13) if f"HijriMain_{i}" in ts.columns]
    hijri_sec_cols  = [f"HijriSec_{i}"  for i in range(1, 13) if f"HijriSec_{i}"  in ts.columns]
    exog_cols = hijri_main_cols + hijri_sec_cols
    has_exog = len(exog_cols) > 0

    model_list = ["SNAIVE", "ETS", "Theta", "Regression", "ARIMA", "SARIMA", "ARIMAX", "SARIMAX"]

    weights = {"MAPE": w_mape, "sMAPE": w_smape, "WAPE": w_wape, "RMSE": w_rmse, "MAE": w_mae, "MASE": w_mase}

    metrics_rows = []
    selection_rows = []
    forecast_rows = []

    subcats = ts["sub_cat"].astype(str).unique().tolist()
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    for i, sc in enumerate(subcats, 1):
        g = ts[ts["sub_cat"].astype(str) == str(sc)].copy().sort_values("date").reset_index(drop=True)

        y = pd.to_numeric(g["y_subcat_vol_kg"], errors="coerce").fillna(0.0).astype(float).values
        y = np.maximum(y, 0.0)

        X = None
        if has_exog:
            X = g[exog_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0).values
            X = np.where(np.isfinite(X), X, 0.0)

        bt = int(fsteps_error)
        if len(y) <= bt + 3:
            y_tr, y_te = y, y[-bt:] if len(y) >= bt else y
            X_tr = X[:-bt] if (X is not None and len(X) > bt) else X
        else:
            y_tr, y_te = y[:-bt], y[-bt:]
            X_tr = None if X is None else X[:-bt]

        # Conservative differencing defaults for sub-cat (can be upgraded later)
        d_val, D_val = 0, 0

        # --- Holdout forecasts for selection metrics
        cand_bt = {
            "SNAIVE": _snaive_forecast(y_tr, bt, seasonality),
            "ETS": _ets_forecast(y_tr, bt, seasonality),
            "Theta": _theta_forecast(y_tr, bt),
            "Regression": _regression_forecast(y_tr, X_tr, bt, seasonality),
            "ARIMA": _fit_forecast_sarimax(y_tr, None, bt, seasonality, d_val, 0, use_exog=False, seasonal=False),
            "SARIMA": _fit_forecast_sarimax(y_tr, None, bt, seasonality, d_val, D_val, use_exog=False, seasonal=True),
            "ARIMAX": _fit_forecast_sarimax(y_tr, X_tr, bt, seasonality, d_val, 0, use_exog=True, seasonal=False) if (X_tr is not None and has_exog) else None,
            "SARIMAX": _fit_forecast_sarimax(y_tr, X_tr, bt, seasonality, d_val, D_val, use_exog=True, seasonal=True) if (X_tr is not None and has_exog) else None,
        }

        # Guarantee numeric forecasts for metric computation (fallback if any model fails)
        for mn in model_list:
            if cand_bt.get(mn) is None:
                cand_bt[mn] = _fallback_forecast(y_tr, bt)

        # Metrics table (per model)
        sc_metrics = []
        for mn in model_list:
            met = _metrics(y_te, cand_bt[mn], y_train_for_mase=y_tr, m=seasonality)
            row = {"sub_cat": sc, "model": mn, "backtest_months": bt, **met}
            metrics_rows.append(row)
            sc_metrics.append(row)

        dfm = pd.DataFrame(sc_metrics)

        # Build weighted selection score (rank-based)
        dfm_scored = _rank_weighted_score(dfm, weights).sort_values(["score", "MAPE", "sMAPE", "RMSE"], ascending=True)

        # Pick Top-N models by the multi-metric score (not only MAPE)
        top_n = int(max(1, top_n_models))
        chosen = dfm_scored["model"].head(top_n).tolist()

        # --- Refit on full history and forecast future for ALL models (auditability)
        last_date = pd.Timestamp(g["date"].max())
        future_dates = _make_future_dates(last_date, fsteps_out)

        y_full, X_full = y, X
        cand_full = {
            "SNAIVE": _snaive_forecast(y_full, fsteps_out, seasonality),
            "ETS": _ets_forecast(y_full, fsteps_out, seasonality),
            "Theta": _theta_forecast(y_full, fsteps_out),
            "Regression": _regression_forecast(y_full, X_full, fsteps_out, seasonality),
            "ARIMA": _fit_forecast_sarimax(y_full, None, fsteps_out, seasonality, d_val, 0, use_exog=False, seasonal=False),
            "SARIMA": _fit_forecast_sarimax(y_full, None, fsteps_out, seasonality, d_val, D_val, use_exog=False, seasonal=True),
            "ARIMAX": _fit_forecast_sarimax(y_full, X_full, fsteps_out, seasonality, d_val, 0, use_exog=True, seasonal=False) if (X_full is not None and has_exog) else None,
            "SARIMAX": _fit_forecast_sarimax(y_full, X_full, fsteps_out, seasonality, d_val, D_val, use_exog=True, seasonal=True) if (X_full is not None and has_exog) else None,
        }
        for mn in model_list:
            if cand_full.get(mn) is None:
                cand_full[mn] = _fallback_forecast(y_full, fsteps_out)

        # FINAL forecast = median across selected Top-N models (per horizon step)
        mat = np.vstack([np.asarray(cand_full[mn], dtype=float) for mn in chosen])
        final_median = np.nanmedian(mat, axis=0)
        final_median = np.where(np.isfinite(final_median), final_median, 0.0)
        final_median = np.maximum(final_median, 0.0)

        # Store selection info (include leading metrics for traceability)
        lead = dfm_scored.head(top_n).copy()
        selection_rows.append({
            "sub_cat": sc,
            "chosen_models_for_median": " | ".join(chosen),
            "top_n_models": top_n,
            "score_best": float(dfm_scored.iloc[0]["score"]),
            "MAPE_best": float(dfm_scored.iloc[0]["MAPE"]),
            "sMAPE_best": float(dfm_scored.iloc[0]["sMAPE"]),
            "WAPE_best": float(dfm_scored.iloc[0]["WAPE"]),
            "RMSE_best": float(dfm_scored.iloc[0]["RMSE"]),
            "MAE_best": float(dfm_scored.iloc[0]["MAE"]),
            "MASE_best": float(dfm_scored.iloc[0]["MASE"]) if np.isfinite(dfm_scored.iloc[0]["MASE"]) else np.nan,
            "weights": f"MAPE={w_mape}, sMAPE={w_smape}, WAPE={w_wape}, RMSE={w_rmse}, MAE={w_mae}, MASE={w_mase}",
        })

        if verbose:
            print(f"[{i}/{len(subcats)}] sub_cat={sc} | chosen={chosen} | best_score={dfm_scored.iloc[0]['score']:.2f} | best_MAPE={dfm_scored.iloc[0]['MAPE']:.2f}")

        # Write forecast rows
        for k in range(int(fsteps_out)):
            row = {"date": future_dates[k], "sub_cat": sc, "FINAL_MEDIAN": float(final_median[k]), "models_used_for_final": " | ".join(chosen)}
            for mn in model_list:
                row[mn] = float(cand_full[mn][k])
            forecast_rows.append(row)

    # Export
    df_forecasts = pd.DataFrame(forecast_rows).sort_values(["sub_cat", "date"]).reset_index(drop=True)
    df_metrics   = pd.DataFrame(metrics_rows).sort_values(["sub_cat", "model"]).reset_index(drop=True)
    df_select    = pd.DataFrame(selection_rows).sort_values(["sub_cat"]).reset_index(drop=True)

    runinfo = pd.DataFrame([
        {"Key": "model_input_path", "Value": model_input_path},
        {"Key": "output_path", "Value": output_path},
        {"Key": "seasonality", "Value": int(seasonality)},
        {"Key": "fsteps_error", "Value": int(fsteps_error)},
        {"Key": "fsteps_out", "Value": int(fsteps_out)},
        {"Key": "top_n_models", "Value": int(max(1, top_n_models))},
        {"Key": "exog_cols_count", "Value": int(len(exog_cols))},
        {"Key": "sklearn_available", "Value": bool(SKLEARN_OK)},
        {"Key": "selection_method", "Value": "weighted_rank_score_then_median_ensemble"},
        {"Key": "weights", "Value": f"MAPE={w_mape}, sMAPE={w_smape}, WAPE={w_wape}, RMSE={w_rmse}, MAE={w_mae}, MASE={w_mase}"},
    ])

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        df_forecasts.to_excel(writer, sheet_name="Forecasts", index=False)
        df_select.to_excel(writer, sheet_name="Selection", index=False)
        df_metrics.to_excel(writer, sheet_name="Metrics", index=False)
        runinfo.to_excel(writer, sheet_name="RunInfo", index=False)

    return {
        "output_path": output_path,
        "forecasts": df_forecasts,
        "selection": df_select,
        "metrics": df_metrics,
    }

## SKU share forecasting and reconciliation

In [ ]:
# Cell 1 — REWRITE (Fix SubCat_TS target column name to match your actual file)
# Your SubCat_TS has: y_subcat_vol_kg (NOT vol_kg)
# Share sheet already has: share_vol

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from math import sqrt

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing, SimpleExpSmoothing

# ---- optional sklearn ----
SKLEARN_OK = True
try:
    from sklearn.linear_model import Ridge
    from sklearn.preprocessing import StandardScaler
except Exception:
    SKLEARN_OK = False


# =========================================================
# Robust helpers
# =========================================================
def _pick_col(df, candidates, required=True):
    """Pick the first existing column from candidates (case-insensitive)."""
    cols = list(df.columns)
    lower_map = {c.lower(): c for c in cols}
    for cand in candidates:
        if cand in cols:
            return cand
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    if required:
        raise ValueError(f"None of these columns exist: {candidates}\nAvailable columns: {cols}")
    return None

def _unwrap_forecast(obj):
    if isinstance(obj, (tuple, list)) and len(obj) >= 1:
        return obj[0]
    return obj

def _to_1d_float(x):
    x = _unwrap_forecast(x)
    if np.isscalar(x):
        return np.asarray([float(x)], dtype=float)
    try:
        return np.asarray(x, dtype=float).reshape(-1)
    except Exception:
        return np.asarray(list(x), dtype=float).reshape(-1)

def _ensure_len(x, h, fill_value=0.0):
    h = int(h)
    a = _to_1d_float(x)
    if len(a) == h:
        return a
    if len(a) == 0:
        return np.full(h, float(fill_value), dtype=float)
    if len(a) > h:
        return a[:h]
    last = float(a[-1]) if np.isfinite(a[-1]) else float(fill_value)
    pad = np.full(h - len(a), last, dtype=float)
    return np.concatenate([a, pad], axis=0)

def _coerce_month_start(s):
    dt = pd.to_datetime(s, errors="coerce")
    if not hasattr(dt, "dt"):
        dt = pd.to_datetime(pd.Series(dt), errors="coerce")
    return dt.dt.to_period("M").dt.start_time

def _make_future_dates(last_date, fsteps):
    last = pd.Timestamp(last_date)
    return pd.date_range(last + pd.offsets.MonthBegin(1), periods=int(fsteps), freq="MS")

def _future_exog_from_history(exog_hist, m, fsteps):
    X = np.asarray(exog_hist, dtype=float)
    if X.ndim == 1:
        X = X.reshape(-1, 1)
    if X.shape[0] == 0:
        return np.zeros((fsteps, X.shape[1]))
    block = X[-min(m, X.shape[0]):, :] if (m and m > 1) else X[-1:, :]
    reps = int(np.ceil(fsteps / block.shape[0]))
    Xf = np.vstack([block] * reps)[:fsteps, :]
    return np.where(np.isfinite(Xf), Xf, 0.0)

def _fallback_forecast(y, h):
    y = np.asarray(y, dtype=float)
    y = y[np.isfinite(y)]
    if len(y) == 0:
        return np.zeros(int(h), dtype=float)
    last = float(y[-1])
    mean = float(np.mean(y))
    val = last if np.isfinite(last) else mean
    if not np.isfinite(val):
        val = 0.0
    return np.maximum(np.repeat(val, int(h)), 0.0)

def _metrics(y_true, y_pred, y_train_for_mase=None, m=1):
    y_true = np.asarray(y_true, dtype=float).reshape(-1)
    y_pred = _ensure_len(y_pred, len(y_true), fill_value=0.0)

    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true, y_pred = y_true[mask], y_pred[mask]
    if len(y_true) == 0:
        return {"MAE": np.nan, "RMSE": np.nan, "MAPE": np.nan, "sMAPE": np.nan, "WAPE": np.nan, "MASE": np.nan}

    eps = 1e-9
    err = y_true - y_pred
    mae = float(np.mean(np.abs(err)))
    rmse = float(sqrt(np.mean(err**2)))
    mape = float(np.mean(np.abs(err) / np.maximum(np.abs(y_true), eps)) * 100.0)
    smape = float(np.mean(2.0 * np.abs(err) / np.maximum(np.abs(y_true) + np.abs(y_pred), eps)) * 100.0)
    wape = float(np.sum(np.abs(err)) / np.maximum(np.sum(np.abs(y_true)), eps) * 100.0)

    mase = np.nan
    if y_train_for_mase is not None:
        ytr = np.asarray(y_train_for_mase, dtype=float).reshape(-1)
        ytr = ytr[np.isfinite(ytr)]
        if len(ytr) > (m if (m and m > 1) else 1):
            if m and m > 1:
                scale = np.mean(np.abs(ytr[m:] - ytr[:-m]))
            else:
                scale = np.mean(np.abs(ytr[1:] - ytr[:-1]))
            if np.isfinite(scale) and scale > 0:
                mase = float(mae / scale)

    return {"MAE": mae, "RMSE": rmse, "MAPE": mape, "sMAPE": smape, "WAPE": wape, "MASE": mase}

def _rank_weighted_score(df_metrics, weights):
    df = df_metrics.copy()
    for c in ["MAPE", "sMAPE", "WAPE", "RMSE", "MAE", "MASE"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df["MASE_fill"] = df["MASE"].copy()
    if df["MASE_fill"].notna().any():
        worst = float(df["MASE_fill"].max())
        df["MASE_fill"] = df["MASE_fill"].fillna(worst + 1.0)
    else:
        df["MASE_fill"] = 1e9

    df["r_MAPE"]  = df["MAPE"].rank(method="min", ascending=True)
    df["r_sMAPE"] = df["sMAPE"].rank(method="min", ascending=True)
    df["r_WAPE"]  = df["WAPE"].rank(method="min", ascending=True)
    df["r_RMSE"]  = df["RMSE"].rank(method="min", ascending=True)
    df["r_MAE"]   = df["MAE"].rank(method="min", ascending=True)
    df["r_MASE"]  = df["MASE_fill"].rank(method="min", ascending=True)

    df["score"] = (
        weights["MAPE"]  * df["r_MAPE"]  +
        weights["sMAPE"] * df["r_sMAPE"] +
        weights["WAPE"]  * df["r_WAPE"]  +
        weights["RMSE"]  * df["r_RMSE"]  +
        weights["MAE"]   * df["r_MAE"]   +
        weights["MASE"]  * df["r_MASE"]
    )
    return df


# =========================================================
# Models
# =========================================================
def _theta_forecast(y, h):
    y = np.asarray(y, dtype=float)
    n = len(y)
    if n < 3:
        return _fallback_forecast(y, h)

    t = np.arange(n, dtype=float)
    A = np.vstack([t, np.ones(n)]).T
    b1, b0 = np.linalg.lstsq(A, y, rcond=None)[0]
    t_f = np.arange(n, n + int(h), dtype=float)
    lin_fc = b1 * t_f + b0

    try:
        ses = SimpleExpSmoothing(y, initialization_method="estimated").fit(optimized=True)
        ses_fc = np.asarray(ses.forecast(int(h)), dtype=float)
    except Exception:
        ses_fc = _fallback_forecast(y, h)

    return 0.5 * (lin_fc + ses_fc)

def _ets_forecast(y, h, m):
    y = np.asarray(y, dtype=float)
    if len(y) < 6:
        return _fallback_forecast(y, h)

    seasonal = "add" if (m and m > 1 and len(y) >= 2*m) else None
    try:
        fit = ExponentialSmoothing(
            y, trend="add", seasonal=seasonal,
            seasonal_periods=m if seasonal else None,
            initialization_method="estimated"
        ).fit(optimized=True)
        return np.asarray(fit.forecast(int(h)), dtype=float)
    except Exception:
        try:
            ses = SimpleExpSmoothing(y, initialization_method="estimated").fit(optimized=True)
            return np.asarray(ses.forecast(int(h)), dtype=float)
        except Exception:
            return _fallback_forecast(y, h)

_GRID_P = (0, 1)
_GRID_Q = (0, 1)
_GRID_Ps = (0, 1)
_GRID_Qs = (0, 1)
_TREND_OPTIONS = ("n", "c")

def _sarimax_grid_search(y, X, m, d, D, seasonal=True):
    best = None
    best_aic = np.inf
    y = np.asarray(y, dtype=float)
    X_use = None if X is None else np.asarray(X, dtype=float)

    for trend in _TREND_OPTIONS:
        for p in _GRID_P:
            for q in _GRID_Q:
                if seasonal:
                    for P in _GRID_Ps:
                        for Q in _GRID_Qs:
                            order = (p, int(d), q)
                            seas  = (P, int(D), Q, int(m))
                            try:
                                res = SARIMAX(
                                    y, exog=X_use, order=order, seasonal_order=seas, trend=trend,
                                    enforce_stationarity=False, enforce_invertibility=False
                                ).fit(disp=False)
                                aic = float(res.aic)
                                if np.isfinite(aic) and aic < best_aic:
                                    best_aic = aic
                                    best = (order, seas, trend, aic)
                            except Exception:
                                pass
                else:
                    order = (p, int(d), q)
                    seas  = (0, 0, 0, 0)
                    try:
                        res = SARIMAX(
                            y, exog=X_use, order=order, seasonal_order=seas, trend=trend,
                            enforce_stationarity=False, enforce_invertibility=False
                        ).fit(disp=False)
                        aic = float(res.aic)
                        if np.isfinite(aic) and aic < best_aic:
                            best_aic = aic
                            best = (order, seas, trend, aic)
                    except Exception:
                        pass
    return best

def _fit_forecast_sarimax(y, X, h, m, d, D, use_exog, seasonal):
    best = _sarimax_grid_search(y, X if use_exog else None, m, d, D, seasonal=seasonal)
    if best is None:
        return None, {"order": None, "seasonal_order": None, "trend": None, "aic": np.nan}

    order, seas, trend, _ = best
    try:
        mod = SARIMAX(
            np.asarray(y, dtype=float),
            exog=None if not use_exog else np.asarray(X, dtype=float),
            order=order, seasonal_order=seas, trend=trend,
            enforce_stationarity=False, enforce_invertibility=False
        )
        res = mod.fit(disp=False)
        if use_exog:
            Xf = _future_exog_from_history(X, m, int(h))
            fc = np.asarray(res.forecast(steps=int(h), exog=Xf), dtype=float)
        else:
            fc = np.asarray(res.forecast(steps=int(h)), dtype=float)
        return fc, {"order": order, "seasonal_order": seas, "trend": trend, "aic": float(res.aic)}
    except Exception:
        return None, {"order": order, "seasonal_order": seas, "trend": trend, "aic": np.nan}

def _regression_forecast(y, X, h, m):
    if not SKLEARN_OK:
        return None, {"note": "sklearn not available"}

    y = np.asarray(y, dtype=float)
    Xex = None if X is None else np.asarray(X, dtype=float)
    if Xex is not None and Xex.ndim == 1:
        Xex = Xex.reshape(-1, 1)

    n = len(y)
    if n < 12:
        return _fallback_forecast(y, h), {"note": "short history fallback"}

    max_lag = min(12, n - 2) if (m and m > 1) else min(6, n - 2)

    rows, tgt = [], []
    for t in range(max_lag, n):
        feats = [y[t - i] for i in range(1, max_lag + 1)]
        feats.append(np.mean(y[max(0, t - m):t]) if (m and m > 1) else np.mean(y[:t]))
        if Xex is not None:
            feats.extend(Xex[t].tolist())
        rows.append(feats)
        tgt.append(y[t])

    Xsup = np.asarray(rows, dtype=float)
    ysup = np.asarray(tgt, dtype=float)
    Xsup = np.where(np.isfinite(Xsup), Xsup, 0.0)
    ysup = np.where(np.isfinite(ysup), ysup, 0.0)

    scaler = StandardScaler()
    Xs = scaler.fit_transform(Xsup)

    model = Ridge(alpha=1.0, random_state=0)
    model.fit(Xs, ysup)

    y_hist = y.copy()
    Xf = None if Xex is None else _future_exog_from_history(Xex, m, int(h))

    fc = []
    for step in range(int(h)):
        t = len(y_hist)
        feats = [y_hist[t - i] for i in range(1, max_lag + 1)]
        feats.append(np.mean(y_hist[max(0, t - m):t]) if (m and m > 1) else np.mean(y_hist))
        if Xf is not None:
            feats.extend(Xf[step].tolist())
        Xrow = np.asarray(feats, dtype=float).reshape(1, -1)
        Xrow = np.where(np.isfinite(Xrow), Xrow, 0.0)
        yhat = float(model.predict(scaler.transform(Xrow))[0])
        fc.append(yhat)
        y_hist = np.append(y_hist, yhat)

    return np.asarray(fc, dtype=float), {"note": f"ridge lag={max_lag}"}


# =========================================================
# Main function
# =========================================================
def SKU_Forecast_From_SubCatAndShare(
    monthly_sales_path: str = r"workspace/SubCatStrategy/Input/monthly_sales.xlsx",
    subcat_path: str       = r"workspace/SubCatStrategy/Input/SubCat.xlsx",
    data_path: str         = r"workspace/SubCatStrategy/Data/Data.xlsx",
    model_input_path: str  = r"workspace/SubCatStrategy/Data/Model_Input.xlsx",
    seasonality: int = 12,
    fsteps_error: int = 6,
    fsteps_out: int = 6,
    top_n_models: int = 3,
    w_mape: float  = 3.0,
    w_smape: float = 2.0,
    w_wape: float  = 2.0,
    w_rmse: float  = 2.0,
    w_mae: float   = 1.0,
    w_mase: float  = 1.0,
    output_path: str = r"workspace/SubCatStrategy/SubCatForecast/SKU_Forecasts_FromSubCatShare.xlsx",
    verbose: bool = True,
) -> dict:

    weights = {"MAPE": w_mape, "sMAPE": w_smape, "WAPE": w_wape, "RMSE": w_rmse, "MAE": w_mae, "MASE": w_mase}

    # 0) Refresh artifacts (Data + Model_Input)
    _ = Build_Model_Input(
        monthly_sales_path=monthly_sales_path,
        subcat_path=subcat_path,
        data_path=data_path,
        model_input_path=model_input_path,
    )

    # ---------------------------------------------------------
    # 1) Load SubCat_TS (your real column: y_subcat_vol_kg)
    # ---------------------------------------------------------
    subcat_ts = pd.read_excel(model_input_path, sheet_name="SubCat_TS")

    c_date = _pick_col(subcat_ts, ["date", "Date_Gregorian", "Date"], required=True)
    c_sub  = _pick_col(subcat_ts, ["sub_cat", "Sub Cat", "SubCat", "SubCategory"], required=True)

    # IMPORTANT FIX: include your real column name y_subcat_vol_kg
    c_y    = _pick_col(
        subcat_ts,
        [
            "y_subcat_vol_kg",      # <-- your actual file
            "subcat_vol_kg",        # sometimes exists
            "vol_kg", "Vol (Kg)", "Vol(Kg)", "Volume",
            "subcat_vol", "SubCat_Vol", "VolKg", "vol"
        ],
        required=True
    )

    subcat_ts = subcat_ts.rename(columns={c_date: "date", c_sub: "sub_cat", c_y: "vol_kg"}).copy()
    subcat_ts["date"] = _coerce_month_start(subcat_ts["date"])
    subcat_ts["sub_cat"] = subcat_ts["sub_cat"].astype(str)
    subcat_ts["vol_kg"] = pd.to_numeric(subcat_ts["vol_kg"], errors="coerce").fillna(0.0).astype(float)
    subcat_ts = subcat_ts.sort_values(["sub_cat", "date"]).reset_index(drop=True)

    # exog (Hijri dummies if exist)
    hijri_main_cols = [f"HijriMain_{i}" for i in range(1, 13) if f"HijriMain_{i}" in subcat_ts.columns]
    hijri_sec_cols  = [f"HijriSec_{i}"  for i in range(1, 13) if f"HijriSec_{i}"  in subcat_ts.columns]
    sub_exog_cols = hijri_main_cols + hijri_sec_cols
    sub_has_exog = len(sub_exog_cols) > 0

    # ---------------------------------------------------------
    # 2) SubCat forecasting (7 models -> top-N -> median)
    # ---------------------------------------------------------
    SUB_MODELS = [
        "Regression(Ridge)", "Theta", "ETS",
        "ARIMA(manual)", "SARIMA(manual)", "ARIMAX(manual)", "SARIMAX(manual)"
    ]

    sub_metrics_rows, sub_selection_rows, sub_forecast_rows = [], [], []
    subcats = subcat_ts["sub_cat"].unique().tolist()

    for i, sc in enumerate(subcats, 1):
        g = subcat_ts[subcat_ts["sub_cat"] == sc].sort_values("date").reset_index(drop=True)

        y = np.maximum(g["vol_kg"].values.astype(float), 0.0)

        X = None
        if sub_has_exog:
            X = g[sub_exog_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0).values
            X = np.where(np.isfinite(X), X, 0.0)

        bt = int(fsteps_error)
        if len(y) <= bt + 4:
            y_tr, y_te = y, y[-bt:] if len(y) >= bt else y
            X_tr = X[:-bt] if (X is not None and len(X) > bt) else X
        else:
            y_tr, y_te = y[:-bt], y[-bt:]
            X_tr = None if X is None else X[:-bt]

        d_val, D_val = 0, 0

        cand_bt = {}
        fc, _ = _regression_forecast(y_tr, X_tr, bt, seasonality)
        cand_bt["Regression(Ridge)"] = fc if fc is not None else _fallback_forecast(y_tr, bt)
        cand_bt["Theta"] = _theta_forecast(y_tr, bt)
        cand_bt["ETS"]   = _ets_forecast(y_tr, bt, seasonality)

        fc, _ = _fit_forecast_sarimax(y_tr, None, bt, seasonality, d_val, 0, use_exog=False, seasonal=False)
        cand_bt["ARIMA(manual)"] = fc if fc is not None else _fallback_forecast(y_tr, bt)
        fc, _ = _fit_forecast_sarimax(y_tr, None, bt, seasonality, d_val, D_val, use_exog=False, seasonal=True)
        cand_bt["SARIMA(manual)"] = fc if fc is not None else _fallback_forecast(y_tr, bt)

        if X_tr is not None and sub_has_exog:
            fc, _ = _fit_forecast_sarimax(y_tr, X_tr, bt, seasonality, d_val, 0, use_exog=True, seasonal=False)
            cand_bt["ARIMAX(manual)"] = fc if fc is not None else _fallback_forecast(y_tr, bt)
            fc, _ = _fit_forecast_sarimax(y_tr, X_tr, bt, seasonality, d_val, D_val, use_exog=True, seasonal=True)
            cand_bt["SARIMAX(manual)"] = fc if fc is not None else _fallback_forecast(y_tr, bt)
        else:
            cand_bt["ARIMAX(manual)"] = _fallback_forecast(y_tr, bt)
            cand_bt["SARIMAX(manual)"] = _fallback_forecast(y_tr, bt)

        for mn in SUB_MODELS:
            cand_bt[mn] = _ensure_len(cand_bt[mn], bt, fill_value=float(y_tr[-1]) if len(y_tr) else 0.0)

        sc_metrics = []
        for mn in SUB_MODELS:
            met = _metrics(y_te, cand_bt[mn], y_train_for_mase=y_tr, m=seasonality)
            row = {"sub_cat": sc, "model": mn, "backtest_months": bt, **met}
            sub_metrics_rows.append(row)
            sc_metrics.append(row)

        dfm = pd.DataFrame(sc_metrics)
        dfm_scored = _rank_weighted_score(dfm, weights).sort_values(["score", "MAPE", "sMAPE", "RMSE"], ascending=True)
        chosen = dfm_scored["model"].head(int(max(1, top_n_models))).tolist()

        last_date = pd.Timestamp(g["date"].max())
        future_dates = _make_future_dates(last_date, fsteps_out)

        cand_full = {}
        fc, _ = _regression_forecast(y, X, fsteps_out, seasonality)
        cand_full["Regression(Ridge)"] = fc if fc is not None else _fallback_forecast(y, fsteps_out)
        cand_full["Theta"] = _theta_forecast(y, fsteps_out)
        cand_full["ETS"]   = _ets_forecast(y, fsteps_out, seasonality)

        fc, _ = _fit_forecast_sarimax(y, None, fsteps_out, seasonality, d_val, 0, use_exog=False, seasonal=False)
        cand_full["ARIMA(manual)"] = fc if fc is not None else _fallback_forecast(y, fsteps_out)
        fc, _ = _fit_forecast_sarimax(y, None, fsteps_out, seasonality, d_val, D_val, use_exog=False, seasonal=True)
        cand_full["SARIMA(manual)"] = fc if fc is not None else _fallback_forecast(y, fsteps_out)

        if X is not None and sub_has_exog:
            fc, _ = _fit_forecast_sarimax(y, X, fsteps_out, seasonality, d_val, 0, use_exog=True, seasonal=False)
            cand_full["ARIMAX(manual)"] = fc if fc is not None else _fallback_forecast(y, fsteps_out)
            fc, _ = _fit_forecast_sarimax(y, X, fsteps_out, seasonality, d_val, D_val, use_exog=True, seasonal=True)
            cand_full["SARIMAX(manual)"] = fc if fc is not None else _fallback_forecast(y, fsteps_out)
        else:
            cand_full["ARIMAX(manual)"] = _fallback_forecast(y, fsteps_out)
            cand_full["SARIMAX(manual)"] = _fallback_forecast(y, fsteps_out)

        for mn in SUB_MODELS:
            cand_full[mn] = np.maximum(_ensure_len(cand_full[mn], fsteps_out, fill_value=float(y[-1]) if len(y) else 0.0), 0.0)

        mat = np.vstack([cand_full[mn] for mn in chosen])
        final_median = np.nanmedian(mat, axis=0)
        final_median = np.maximum(np.where(np.isfinite(final_median), final_median, 0.0), 0.0)

        sub_selection_rows.append({
            "sub_cat": sc,
            "chosen_models_for_median": " | ".join(chosen),
            "top_n_models": int(max(1, top_n_models)),
            "best_score": float(dfm_scored.iloc[0]["score"]) if len(dfm_scored) else np.nan,
            "best_MAPE": float(dfm_scored.iloc[0]["MAPE"]) if len(dfm_scored) else np.nan,
        })

        for k in range(int(fsteps_out)):
            r = {"date": future_dates[k], "sub_cat": sc, "SUBCAT_FINAL_MEDIAN": float(final_median[k]), "models_used_for_median": " | ".join(chosen)}
            for mn in SUB_MODELS:
                r[mn] = float(cand_full[mn][k])
            sub_forecast_rows.append(r)

        if verbose and (i % 5 == 0 or i == len(subcats)):
            print(f"SubCat forecasts: {i}/{len(subcats)} processed")

    df_subcat_fc = pd.DataFrame(sub_forecast_rows)
    df_subcat_fc["date"] = _coerce_month_start(df_subcat_fc["date"])
    df_subcat_fc["sub_cat"] = df_subcat_fc["sub_cat"].astype(str)
    df_subcat_final = df_subcat_fc[["date", "sub_cat", "SUBCAT_FINAL_MEDIAN"]].copy()

    # ---------------------------------------------------------
    # 3) Share forecasting (your file: share_vol exists)
    # ---------------------------------------------------------
    share_ts = pd.read_excel(model_input_path, sheet_name="SKU_Share_TS")

    c_date = _pick_col(share_ts, ["date", "Date_Gregorian", "Date"], required=True)
    c_sub  = _pick_col(share_ts, ["sub_cat", "Sub Cat", "SubCat", "SubCategory"], required=True)
    c_sku  = _pick_col(share_ts, ["material_code", "Material Code", "Material_Code", "SKU", "sku"], required=True)
    c_sh   = _pick_col(share_ts, ["share_vol", "share", "Share", "share_kg", "share_ratio", "share_vol_kg"], required=True)

    share_ts = share_ts.rename(columns={c_date:"date", c_sub:"sub_cat", c_sku:"material_code", c_sh:"share_vol"}).copy()
    share_ts["date"] = _coerce_month_start(share_ts["date"])
    share_ts["sub_cat"] = share_ts["sub_cat"].astype(str)
    share_ts["material_code"] = share_ts["material_code"].astype(str)
    share_ts["share_vol"] = pd.to_numeric(share_ts["share_vol"], errors="coerce").fillna(0.0).astype(float)
    share_ts["share_vol"] = np.clip(share_ts["share_vol"].values, 0.0, 1.0)
    share_ts = share_ts.dropna(subset=["date", "sub_cat", "material_code"]).sort_values(["sub_cat","material_code","date"]).reset_index(drop=True)

    hijri_main_cols = [f"HijriMain_{i}" for i in range(1, 13) if f"HijriMain_{i}" in share_ts.columns]
    hijri_sec_cols  = [f"HijriSec_{i}"  for i in range(1, 13) if f"HijriSec_{i}"  in share_ts.columns]
    exog_cols = hijri_main_cols + hijri_sec_cols
    has_exog = len(exog_cols) > 0

    SHARE_MODELS = [
        "Regression(Ridge)", "Theta", "ETS",
        "ARIMA(manual)", "SARIMA(manual)", "ARIMAX(manual)", "SARIMAX(manual)"
    ]

    share_metrics_rows, share_selection_rows, share_raw_rows = [], [], []
    groups = share_ts.groupby(["sub_cat", "material_code"], sort=False)
    total_groups = len(groups)

    for gi, ((sc, sku), g) in enumerate(groups, 1):
        g = g.sort_values("date").reset_index(drop=True)

        y = np.clip(g["share_vol"].values.astype(float), 0.0, 1.0)

        X = None
        if has_exog:
            X = g[exog_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0).values
            X = np.where(np.isfinite(X), X, 0.0)

        bt = int(fsteps_error)
        if len(y) <= bt + 4:
            y_tr, y_te = y, y[-bt:] if len(y) >= bt else y
            X_tr = X[:-bt] if (X is not None and len(X) > bt) else X
        else:
            y_tr, y_te = y[:-bt], y[-bt:]
            X_tr = None if X is None else X[:-bt]

        d_val, D_val = 0, 0

        cand_bt = {}
        fc, _ = _regression_forecast(y_tr, X_tr, bt, seasonality)
        cand_bt["Regression(Ridge)"] = fc if fc is not None else _fallback_forecast(y_tr, bt)
        cand_bt["Theta"] = _theta_forecast(y_tr, bt)
        cand_bt["ETS"]   = _ets_forecast(y_tr, bt, seasonality)

        fc, _ = _fit_forecast_sarimax(y_tr, None, bt, seasonality, d_val, 0, use_exog=False, seasonal=False)
        cand_bt["ARIMA(manual)"] = fc if fc is not None else _fallback_forecast(y_tr, bt)
        fc, _ = _fit_forecast_sarimax(y_tr, None, bt, seasonality, d_val, D_val, use_exog=False, seasonal=True)
        cand_bt["SARIMA(manual)"] = fc if fc is not None else _fallback_forecast(y_tr, bt)

        if X_tr is not None and has_exog:
            fc, _ = _fit_forecast_sarimax(y_tr, X_tr, bt, seasonality, d_val, 0, use_exog=True, seasonal=False)
            cand_bt["ARIMAX(manual)"] = fc if fc is not None else _fallback_forecast(y_tr, bt)
            fc, _ = _fit_forecast_sarimax(y_tr, X_tr, bt, seasonality, d_val, D_val, use_exog=True, seasonal=True)
            cand_bt["SARIMAX(manual)"] = fc if fc is not None else _fallback_forecast(y_tr, bt)
        else:
            cand_bt["ARIMAX(manual)"] = _fallback_forecast(y_tr, bt)
            cand_bt["SARIMAX(manual)"] = _fallback_forecast(y_tr, bt)

        for mn in SHARE_MODELS:
            cand_bt[mn] = np.clip(_ensure_len(cand_bt[mn], bt, fill_value=0.0), 0.0, 1.0)

        sku_metrics = []
        for mn in SHARE_MODELS:
            met = _metrics(y_te, cand_bt[mn], y_train_for_mase=y_tr, m=seasonality)
            row = {"sub_cat": sc, "material_code": sku, "model": mn, "backtest_months": bt, **met}
            share_metrics_rows.append(row)
            sku_metrics.append(row)

        dfm = pd.DataFrame(sku_metrics)
        dfm_scored = _rank_weighted_score(dfm, weights).sort_values(["score", "MAPE", "sMAPE", "RMSE"], ascending=True)
        chosen = dfm_scored["model"].head(int(max(1, top_n_models))).tolist()

        last_date = pd.Timestamp(g["date"].max())
        future_dates = _make_future_dates(last_date, fsteps_out)

        cand_full = {}
        fc, _ = _regression_forecast(y, X, fsteps_out, seasonality)
        cand_full["Regression(Ridge)"] = fc if fc is not None else _fallback_forecast(y, fsteps_out)
        cand_full["Theta"] = _theta_forecast(y, fsteps_out)
        cand_full["ETS"]   = _ets_forecast(y, fsteps_out, seasonality)

        fc, _ = _fit_forecast_sarimax(y, None, fsteps_out, seasonality, d_val, 0, use_exog=False, seasonal=False)
        cand_full["ARIMA(manual)"] = fc if fc is not None else _fallback_forecast(y, fsteps_out)
        fc, _ = _fit_forecast_sarimax(y, None, fsteps_out, seasonality, d_val, D_val, use_exog=False, seasonal=True)
        cand_full["SARIMA(manual)"] = fc if fc is not None else _fallback_forecast(y, fsteps_out)

        if X is not None and has_exog:
            fc, _ = _fit_forecast_sarimax(y, X, fsteps_out, seasonality, d_val, 0, use_exog=True, seasonal=False)
            cand_full["ARIMAX(manual)"] = fc if fc is not None else _fallback_forecast(y, fsteps_out)
            fc, _ = _fit_forecast_sarimax(y, X, fsteps_out, seasonality, d_val, D_val, use_exog=True, seasonal=True)
            cand_full["SARIMAX(manual)"] = fc if fc is not None else _fallback_forecast(y, fsteps_out)
        else:
            cand_full["ARIMAX(manual)"] = _fallback_forecast(y, fsteps_out)
            cand_full["SARIMAX(manual)"] = _fallback_forecast(y, fsteps_out)

        for mn in SHARE_MODELS:
            cand_full[mn] = np.clip(_ensure_len(cand_full[mn], fsteps_out, fill_value=0.0), 0.0, 1.0)

        mat = np.vstack([cand_full[mn] for mn in chosen])
        share_median_raw = np.nanmedian(mat, axis=0)
        share_median_raw = np.clip(np.where(np.isfinite(share_median_raw), share_median_raw, 0.0), 0.0, 1.0)

        share_selection_rows.append({
            "sub_cat": sc,
            "material_code": sku,
            "chosen_models_for_median": " | ".join(chosen),
            "top_n_models": int(max(1, top_n_models)),
            "best_score": float(dfm_scored.iloc[0]["score"]) if len(dfm_scored) else np.nan,
            "best_MAPE": float(dfm_scored.iloc[0]["MAPE"]) if len(dfm_scored) else np.nan,
        })

        for k in range(int(fsteps_out)):
            r = {
                "date": future_dates[k],
                "sub_cat": sc,
                "material_code": sku,
                "SHARE_MEDIAN_RAW": float(share_median_raw[k]),
                "models_used_for_median": " | ".join(chosen),
            }
            for mn in SHARE_MODELS:
                r[mn] = float(cand_full[mn][k])
            share_raw_rows.append(r)

        if verbose and (gi % 30 == 0 or gi == total_groups):
            print(f"Share forecasts: {gi}/{total_groups} groups processed")

    df_share_raw = pd.DataFrame(share_raw_rows)
    df_share_raw["date"] = _coerce_month_start(df_share_raw["date"])
    df_share_raw["sub_cat"] = df_share_raw["sub_cat"].astype(str)
    df_share_raw["material_code"] = df_share_raw["material_code"].astype(str)

    # ---------------------------------------------------------
    # 4) Normalize share per (sub_cat, date) so sum=1
    # ---------------------------------------------------------
    grp_sum = df_share_raw.groupby(["sub_cat", "date"])["SHARE_MEDIAN_RAW"].transform("sum").replace([0, np.inf, -np.inf], np.nan)
    grp_size = df_share_raw.groupby(["sub_cat", "date"])["material_code"].transform("count").astype(float).replace(0, np.nan)

    share_norm = np.where(np.isfinite(grp_sum), df_share_raw["SHARE_MEDIAN_RAW"].values / grp_sum.values, np.nan)
    share_norm = np.where(np.isfinite(share_norm), share_norm, 1.0 / grp_size.values)

    df_share_raw["SHARE_MEDIAN_NORM"] = np.clip(share_norm, 0.0, 1.0)

    grp_sum2 = df_share_raw.groupby(["sub_cat", "date"])["SHARE_MEDIAN_NORM"].transform("sum").replace(0, np.nan)
    df_share_raw["SHARE_MEDIAN_NORM"] = np.where(
        np.isfinite(grp_sum2),
        df_share_raw["SHARE_MEDIAN_NORM"].values / grp_sum2.values,
        df_share_raw["SHARE_MEDIAN_NORM"].values
    )
    df_share_raw["SHARE_MEDIAN_NORM"] = np.clip(df_share_raw["SHARE_MEDIAN_NORM"].values, 0.0, 1.0)

    df_share_norm = df_share_raw[["date", "sub_cat", "material_code", "SHARE_MEDIAN_NORM", "models_used_for_median"]].copy()

    # ---------------------------------------------------------
    # 5) Final SKU forecast = Share_norm * SubCat_FINAL_MEDIAN
    # ---------------------------------------------------------
    df_final = df_share_norm.merge(df_subcat_final, on=["date", "sub_cat"], how="left")
    df_final["SKU_FORECAST_KG"] = df_final["SHARE_MEDIAN_NORM"] * df_final["SUBCAT_FINAL_MEDIAN"]
    df_final["SKU_FORECAST_KG"] = np.maximum(pd.to_numeric(df_final["SKU_FORECAST_KG"], errors="coerce").fillna(0.0).astype(float), 0.0)

    qa_sum = df_final.groupby(["sub_cat", "date"]).agg(
        sum_share=("SHARE_MEDIAN_NORM", "sum"),
        sum_sku_forecast=("SKU_FORECAST_KG", "sum"),
        subcat_anchor=("SUBCAT_FINAL_MEDIAN", "first"),
    ).reset_index()
    qa_sum["abs_diff_sku_vs_subcat"] = np.abs(qa_sum["sum_sku_forecast"] - qa_sum["subcat_anchor"])
    qa_sum["abs_diff_share_vs_1"] = np.abs(qa_sum["sum_share"] - 1.0)

    # ---------------------------------------------------------
    # 6) Export
    # ---------------------------------------------------------
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    df_subcat_metrics = pd.DataFrame(sub_metrics_rows).sort_values(["sub_cat", "model"]).reset_index(drop=True)
    df_subcat_select  = pd.DataFrame(sub_selection_rows).sort_values(["sub_cat"]).reset_index(drop=True)
    df_share_metrics  = pd.DataFrame(share_metrics_rows).sort_values(["sub_cat", "material_code", "model"]).reset_index(drop=True)
    df_share_select   = pd.DataFrame(share_selection_rows).sort_values(["sub_cat", "material_code"]).reset_index(drop=True)

    df_subcat_fc_out  = df_subcat_fc.sort_values(["sub_cat", "date"]).reset_index(drop=True)
    df_share_raw_out  = df_share_raw.sort_values(["sub_cat", "material_code", "date"]).reset_index(drop=True)
    df_final_out      = df_final.sort_values(["sub_cat", "material_code", "date"]).reset_index(drop=True)

    runinfo = pd.DataFrame([
        {"Key": "model_input_path", "Value": model_input_path},
        {"Key": "data_path", "Value": data_path},
        {"Key": "output_path", "Value": output_path},
        {"Key": "seasonality", "Value": int(seasonality)},
        {"Key": "fsteps_error", "Value": int(fsteps_error)},
        {"Key": "fsteps_out", "Value": int(fsteps_out)},
        {"Key": "top_n_models", "Value": int(max(1, top_n_models))},
        {"Key": "weights", "Value": f"MAPE={w_mape}, sMAPE={w_smape}, WAPE={w_wape}, RMSE={w_rmse}, MAE={w_mae}, MASE={w_mase}"},
        {"Key": "sklearn_available", "Value": bool(SKLEARN_OK)},
        {"Key": "qa_max_abs_diff_sku_vs_subcat", "Value": float(qa_sum["abs_diff_sku_vs_subcat"].max()) if len(qa_sum) else np.nan},
        {"Key": "qa_max_abs_diff_share_vs_1", "Value": float(qa_sum["abs_diff_share_vs_1"].max()) if len(qa_sum) else np.nan},
    ])

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        df_subcat_fc_out.to_excel(writer, sheet_name="SubCat_Forecasts", index=False)
        df_subcat_metrics.to_excel(writer, sheet_name="SubCat_Metrics", index=False)
        df_subcat_select.to_excel(writer, sheet_name="SubCat_Selection", index=False)

        df_share_metrics.to_excel(writer, sheet_name="Share_Metrics", index=False)
        df_share_select.to_excel(writer, sheet_name="Share_Selection", index=False)
        df_share_raw_out.to_excel(writer, sheet_name="Share_Forecasts_Raw", index=False)
        df_share_norm.sort_values(["sub_cat", "material_code", "date"]).to_excel(writer, sheet_name="Share_Forecasts_Norm", index=False)

        df_final_out.to_excel(writer, sheet_name="SKU_Forecasts_Final", index=False)
        qa_sum.sort_values(["sub_cat", "date"]).to_excel(writer, sheet_name="QA_Reconciliation", index=False)
        runinfo.to_excel(writer, sheet_name="RunInfo", index=False)

    return {
        "output_path": output_path,
        "qa_reconciliation": qa_sum,
        "sku_forecasts_final": df_final_out,
    }

## Run hierarchical forecast

In [ ]:
# Cell 2 — Run

res = SKU_Forecast_From_SubCatAndShare(
    seasonality=12,
    fsteps_error=6,
    fsteps_out=6,
    top_n_models=3,
    w_mape=3.0,
    w_smape=2.0,
    w_wape=2.0,
    w_rmse=2.0,
    w_mae=1.0,
    w_mase=1.0,
    output_path=r"workspace/SubCatStrategy/SubCatForecast/SKU_Forecasts_FromSubCatShare.xlsx",
    verbose=True
)

print("Saved:", res["output_path"])
display(res["qa_reconciliation"].head(20))
res["sku_forecasts_final"].head(10)